# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhanu086/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis + time window

One row represents one content page for one month. I will use data from March 2026 as my analysis window because it is a mid-panel month recommended for feature development. My goal is to predict whether a content page should be prioritized for improvement based on its historical performance.

In [45]:
import pandas as pd
import duckdb

In [46]:
import os
os.listdir()

['.config', 'sample_data']

In [47]:
!pip install duckdb huggingface_hub

In [48]:
import duckdb
import pandas as pd
from huggingface_hub import login

In [49]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token)

In [50]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

files[:10]

['.gitattributes',
 'README.md',
 'dim_clients.parquet',
 'dim_content.parquet',
 'fact_content_daily_performance/month=2025-01/data_0.parquet',
 'fact_content_daily_performance/month=2025-02/data_0.parquet',
 'fact_content_daily_performance/month=2025-03/data_0.parquet',
 'fact_content_daily_performance/month=2025-04/data_0.parquet',
 'fact_content_daily_performance/month=2025-05/data_0.parquet',
 'fact_content_daily_performance/month=2025-06/data_0.parquet']

In [51]:
df.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events'],
      dtype='object')

In [52]:
from huggingface_hub import hf_hub_download
import pandas as pd

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

df = pd.read_parquet(file_path)

df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
print("Classification task selected")

Classification task selected


In [54]:
# @title
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features
- Clicks
- Impressions
- CTR
- Average Position
- Published Date

### Label
- Whether the content page should be prioritized for improvement.

### Context
- Month
- Content ID

### Excluded
- Future performance metrics because they are not available at prediction time and would cause data leakage.

In [55]:
import os
os.listdir()

['.config', 'sample_data']

In [56]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [57]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [58]:
# Query 1: Grain check
# Verify one row = one content page on one date

grain_check = df.groupby(
    ["content_hash_id", "report_date"]
).size()

grain_check.value_counts().head()

,count
1,9841378


In [59]:
# Query 2: Row count and date range

print("Number of rows:", len(df))

print("Start date:", df["report_date"].min())

print("End date:", df["report_date"].max())

Number of rows: 9841378
Start date: 2026-03-01
End date: 2026-03-31


In [60]:
import duckdb

duckdb.sql("""
SELECT
    COUNT(*) AS available_rows
FROM df
WHERE gsc_data_available IS TRUE
""")

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘

## Five Features

### 1. GSC Clicks
Available at decision time because historical clicks are recorded before making a prediction.

### 2. GSC Impressions
Available at decision time because search impressions are collected before the prediction moment.

### 3. GSC Average Position
Available at decision time because historical ranking position is already available.

### 4. GA4 Pageviews
Available at decision time because previous page views are recorded from analytics data.

### 5. Content Age
Available at decision time because the page creation date is known before prediction.

In [61]:
features = df[
    [
        "gsc_clicks",
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_pageviews"
    ]
].copy()

features.head()

,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_pageviews
0,0,20,3.350000,NaN
1,0,1,0.000000,NaN
2,1,125,4.928000,NaN
3,0,7,4.000000,NaN
4,0,11,2.272727,NaN


## Leakage Trap

To demonstrate data leakage, I added a future-derived feature that would not be available at prediction time.

The model performance would improve artificially because the feature contains information from the future outcome.

After identifying the leakage, the feature was removed to keep the evaluation honest.

In [62]:
# Create an intentional leakage feature (future information)

features["future_label_signal"] = df["ga4_sessions"]

features.head()

,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_pageviews,future_label_signal
0,0,20,3.350000,NaN,NaN
1,0,1,0.000000,NaN,NaN
2,1,125,4.928000,NaN,NaN
3,0,7,4.000000,NaN,NaN
4,0,11,2.272727,NaN,NaN


In [63]:
# Remove leaked feature

features = features.drop(columns=["future_label_signal"])

features.head()

,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_pageviews
0,0,20,3.350000,NaN
1,0,1,0.000000,NaN
2,1,125,4.928000,NaN
3,0,7,4.000000,NaN
4,0,11,2.272727,NaN


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

This dataset cannot explain why users clicked or ignored a page. It only contains historical search performance. External factors such as seasonality, news events, or website design changes are not captured. Therefore, predictions should support decisions rather than replace human judgment.

In [64]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.